# Module 04 — Region Proposals

Before neural networks could detect objects end-to-end, region proposals were
the critical first stage.  We explore sliding windows, image pyramids, selective
search, and NMS.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from selective_search import nms, soft_nms, recall_vs_proposals, visualize_proposals
%matplotlib inline

## 1. Sliding Window Cost

Let us calculate how many evaluations a sliding window detector needs.

In [ ]:
def sliding_window_count(img_h, img_w, win_h, win_w, stride=1):
    n_h = (img_h - win_h) // stride + 1
    n_w = (img_w - win_w) // stride + 1
    return n_h * n_w

img_size = 500
for win, stride in [(32,8),(64,16),(128,32)]:
    n = sliding_window_count(img_size, img_size, win, win, stride)
    print(f'Window {win}x{win}, stride {stride}: {n:,} positions')

print('\nWith 5 pyramid scales and 3 aspect ratios:')
total = 0
for scale in [1.0, 0.75, 0.5, 0.33, 0.25]:
    for ar_h, ar_w in [(1,1),(1,2),(2,1)]:
        h = int(500*scale*ar_h); w = int(500*scale*ar_w)
        if h < 32 or w < 32: continue
        n = sliding_window_count(500, 500, min(h,64), min(w,64), 16)
        total += n
print(f'Total: ~{total:,} windows')

## 2. Image Pyramids

A Gaussian pyramid is built by blurring then subsampling.

In [ ]:
# Create a synthetic test image
img = np.zeros((256, 256, 3), dtype=np.uint8)
cv2.rectangle(img, (50,50), (200,200), (0,128,255), -1)
cv2.circle(img, (128,128), 40, (255,255,0), -1)

# Build Gaussian pyramid
def gaussian_pyramid(image, n_levels=4):
    pyramid = [image]
    for _ in range(n_levels - 1):
        image = cv2.pyrDown(image)
        pyramid.append(image)
    return pyramid

pyramid = gaussian_pyramid(img, 4)
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, level in zip(axes, pyramid):
    ax.imshow(cv2.cvtColor(level, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{level.shape[1]}x{level.shape[0]}')
    ax.axis('off')
plt.suptitle('Gaussian Pyramid')
plt.tight_layout(); plt.show()

## 3. Non-Maximum Suppression

NMS deduplicates overlapping detections.

In [ ]:
# Synthetic overlapping boxes
boxes = np.array([
    [10, 10, 80, 80], [15, 12, 82, 78], [14, 11, 79, 81],
    [200, 50, 280, 130], [205, 52, 275, 128],
    [100, 150, 180, 230],
], dtype=np.float32)
scores = np.array([0.95, 0.87, 0.80, 0.91, 0.75, 0.60])

kept = nms(boxes, scores, iou_threshold=0.5)
print('Boxes before NMS:', len(boxes))
print('Boxes after NMS: ', len(kept), '→ indices', kept)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for ax, title, sel_boxes, sel_scores in [
    (ax1, 'All detections', boxes, scores),
    (ax2, 'After NMS', boxes[kept], scores[kept])
]:
    ax.set_xlim(0,300); ax.set_ylim(260, 0); ax.set_aspect('equal')
    for (x1,y1,x2,y2), s in zip(sel_boxes, sel_scores):
        ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,linewidth=2,edgecolor='red',facecolor='none'))
        ax.text(x1, y1-3, f'{s:.2f}', color='red', fontsize=9)
    ax.set_title(title)
plt.tight_layout(); plt.show()

## Exercise — IoU Calculator

Implement `compute_iou(box_a, box_b)` for two boxes in xyxy format.
Then use it to verify your NMS implementation produces valid results.

In [ ]:
### EXERCISE
def compute_iou(box_a, box_b):
    """
    Compute IoU between two boxes in xyxy format.
    box_a, box_b: array-like of length 4 (x1, y1, x2, y2).
    Returns float in [0, 1].
    """
    # TODO: implement
    raise NotImplementedError